In [ ]:
from IPython.display import HTML
display(HTML("<style>.rendered_html { font-size: 1.3em; } .code_cell .input_area { font-size: 1.1em; }</style>"))

# 6.1 Special Topics: Additional Boosting Algorithms
### *Optional Module — Not Assessed*
- AdaBoost, LightGBM, CatBoost
- How they differ from XGBoost
- When you might choose them

## AdaBoost: The Original (1996)
- Trains weak learners (decision stumps) sequentially
- Increases weights of misclassified samples
- Historical importance — largely superseded by gradient boosting

## Let's Try AdaBoost

In [ ]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import roc_auc_score
import pandas as pd
import numpy as np

train_df = pd.read_csv('../data/training.csv')
test_df = pd.read_csv('../data/testing.csv')
train_df['DEPARTED'] = (train_df['SEM_3_STATUS'] != 'E').astype(int)
test_df['DEPARTED'] = (test_df['SEM_3_STATUS'] != 'E').astype(int)

numeric_features = ['HS_GPA','HS_MATH_GPA','HS_ENGL_GPA','UNITS_ATTEMPTED_1','UNITS_ATTEMPTED_2',
    'UNITS_COMPLETED_1','UNITS_COMPLETED_2','DFW_UNITS_1','DFW_UNITS_2','GPA_1','GPA_2',
    'DFW_RATE_1','DFW_RATE_2','GRADE_POINTS_1','GRADE_POINTS_2']
categorical_features = ['RACE_ETHNICITY','GENDER','FIRST_GEN_STATUS','COLLEGE']

train_enc = pd.get_dummies(train_df[numeric_features + categorical_features], columns=categorical_features, drop_first=True)
test_enc = pd.get_dummies(test_df[numeric_features + categorical_features], columns=categorical_features, drop_first=True)
train_enc, test_enc = train_enc.align(test_enc, join='left', axis=1, fill_value=0)
train_enc = train_enc.fillna(train_enc.median())
test_enc = test_enc.fillna(test_enc.median())

X_train, y_train = train_enc, train_df['DEPARTED']
X_test, y_test = test_enc, test_df['DEPARTED']

ada = AdaBoostClassifier(n_estimators=200, learning_rate=0.1, random_state=42)
ada.fit(X_train, y_train)
ada_prob = ada.predict_proba(X_test)[:, 1]

print(f"AdaBoost ROC-AUC: {roc_auc_score(y_test, ada_prob):.4f}")

## LightGBM: Fast Gradient Boosting (Microsoft, 2017)
- **Leaf-wise** tree growth (vs. level-wise in XGBoost)
- Histogram-based splitting for speed
- Best for very large datasets (millions of rows)
- Often faster than XGBoost with similar accuracy

In [ ]:
try:
    from lightgbm import LGBMClassifier

    lgbm = LGBMClassifier(
        n_estimators=150, learning_rate=0.1, max_depth=5,
        num_leaves=31, min_child_samples=20,
        subsample=0.8, colsample_bytree=0.8,
        class_weight='balanced', random_state=42, verbose=-1
    )
    lgbm.fit(X_train, y_train)
    lgbm_prob = lgbm.predict_proba(X_test)[:, 1]

    print(f"LightGBM ROC-AUC: {roc_auc_score(y_test, lgbm_prob):.4f}")

except ImportError:
    print("LightGBM not installed. Install with: pip install lightgbm")

## CatBoost: Native Categorical Handling (Yandex, 2017)
- No need for one-hot encoding!
- Ordered boosting prevents target leakage
- Symmetric trees for faster prediction
- Best when you have many categorical features

In [ ]:
try:
    from catboost import CatBoostClassifier

    cat = CatBoostClassifier(
        iterations=150, learning_rate=0.1, depth=5,
        auto_class_weights='Balanced',
        random_seed=42, verbose=0
    )
    cat.fit(X_train, y_train)
    cat_prob = cat.predict_proba(X_test)[:, 1]

    print(f"CatBoost ROC-AUC: {roc_auc_score(y_test, cat_prob):.4f}")

except ImportError:
    print("CatBoost not installed. Install with: pip install catboost")

## Comparison
| Feature | AdaBoost | XGBoost | LightGBM | CatBoost |
|:--------|:---------|:--------|:---------|:---------|
| Year | 1996 | 2014 | 2017 | 2017 |
| Speed | Moderate | Fast | Fastest | Fast |
| Categoricals | Needs encoding | Needs encoding | Needs encoding | Native |
| Best for | Legacy | General | Large data | Categorical-heavy |

## Key Takeaways
- **AdaBoost**: Historically important, largely superseded
- **LightGBM**: Choose for very large datasets or speed
- **CatBoost**: Choose for many categorical features
- **For this course**: XGBoost remains the recommended default
- **Next:** 6.2 Neural Networks